## Exploring predictions

In [ ]:
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import scipy
import methods

import rasterio
from rasterio.windows import Window
from rasterio.transform import Affine

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
# print(tf.config.list_physical_devices('GPU'))

In [ ]:
directory_paths = methods.get_directories()
SAVE_MODEL_DIRECTORY = directory_paths["save_model_dir"]
DATA_DIRECTORY = directory_paths["data_dir"]
FIGURE_DIRECTORY = directory_paths["figure_dir"]
PREDICTIONS_DIRECTORY = directory_paths["predictions_dir"]

## Re-save HII files to smaller values

In [ ]:
error('stop here')

In [ ]:
# for year in (2015, 2016, 2017, 2018, 2019, 2020):
for year in (2018, 2019, 2020):
    
    filename = DATA_DIRECTORY + "hii_" + str(year) + "-01-01.tif"
    with rasterio.open(filename) as orig_tiff:

        lon0, lat0 = orig_tiff.xy(0,0)
        lon1, lat1 = orig_tiff.xy(orig_tiff.shape[0],orig_tiff.shape[1])    
        print(lat0, lat1, lon0, lon1)
        hfi = orig_tiff.read(1)
        meta_data = orig_tiff.meta.copy()

    NO_DATA = 255
    hfi = np.asarray(np.where(hfi!=-32768, np.round(hfi / 6400. * 100), NO_DATA), dtype="uint8")
    meta_data.update({"dtype": hfi.dtype, 
                      "nodata": NO_DATA, 
                      "compress": "lzw"},
                      )
    print(meta_data)
    print(np.min(hfi), np.max(hfi))

    print("   saving the tif file for " + str(year) + "...")
    with rasterio.open("/Users/eabarnes/Documents/hii_" + str(year) + "-01-01_uint8.tif", "w", **meta_data) as dst:
        dst.write(hfi, 1)

## Make coastal buffer file

In [ ]:
error('stop here')

In [ ]:
filename = DATA_DIRECTORY + "hii_2020-01-01.tif"
with rasterio.open(filename) as orig_tiff:

    lon0, lat0 = orig_tiff.xy(0,0)
    lon1, lat1 = orig_tiff.xy(orig_tiff.shape[0],orig_tiff.shape[1])    

    print(lat0, lat1, lon0, lon1)

    mask = orig_tiff.read_masks(1)
    meta_data = orig_tiff.meta.copy()

In [ ]:
# blend the coastal areas and count those as "non-ocean" areas.
hfi_coastal = scipy.ndimage.gaussian_filter(np.asarray(mask,"float32"), 3, mode='wrap')
hfi_coastal = np.where(hfi_coastal!=0, 1., 0.)
hfi_coastal = np.asarray(hfi_coastal, dtype="uint8")

meta_data.update({"dtype": hfi_coastal.dtype, 
                  "nodata": None,
                  "compress": "lzw"},
                  )

print("   saving the tif file ...")
with rasterio.open(DATA_DIRECTORY + "hii_coastal_buffer_mask.tif", "w", **meta_data) as dst:
    dst.write(hfi_coastal, 1)
